## Embed the Wikipedia Movie Plots dataset and then based on a query find the most similar movie and then a Little Qwen-Model to be a expert in movies using some Form of RAG


In [65]:
# Hugging-Face
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from sentence_transformers import SentenceTransformer

# Torch
import torch

# Kaggle import libraries
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Utils
import pickle

In [66]:
file_path = "wiki_movie_plots_deduped.csv"

# Load the latest version
hf_dataset = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    "jrobischon/wikipedia-movie-plots",
    file_path,
)

print("Hugging Face Dataset:", hf_dataset)

/var/folders/d8/k55r2d1s5k78lrh9j8l3lfzw0000gn/T/ipykernel_20977/2763955198.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  hf_dataset = kagglehub.load_dataset(


Hugging Face Dataset: Dataset({
    features: ['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot'],
    num_rows: 34886
})


In [67]:
hf_dataset["Plot"][1]

"The moon, painted with a smiling face hangs over a park at night. A young couple walking past a fence learn on a railing and look up. The moon smiles. They embrace, and the moon's smile gets bigger. They then sit down on a bench by a tree. The moon's view is blocked, causing him to frown. In the last scene, the man fans the woman with his hat because the moon has left the sky and is perched over her shoulder to see everything better."

In [68]:
hf_dataset.remove_columns(
    [
        "Release Year",
        "Origin/Ethnicity",
        "Director",
        "Cast",
        "Genre",
        "Wiki Page",
    ]
)

Dataset({
    features: ['Title', 'Plot'],
    num_rows: 34886
})

In [69]:
model_id = "all-MiniLM-L6-v2"
device = "mps"

In [70]:
encoding_model = SentenceTransformer(model_id)
# embeddings = model.encode(hf_dataset['Plot'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [71]:
embeddings = encoding_model.encode(hf_dataset["Plot"])

In [74]:
pairs = list(zip(hf_dataset["Title"], hf_dataset["Plot"], embeddings))

In [73]:
pairs[0][0], pairs[0][1]

('Kansas Saloon Smashers',
 "A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]")

In [12]:
pickle.dump(pairs, open("pairs_of_embedding.pkl", "wb"))

In [62]:
embeddings = pickle.load(open("pairs_of_embedding.pkl", "rb"))

In [75]:
def find_similar_movie(sentence: str) -> list[tuple]:
    sentence_embedding = encoding_model.encode(sentence)
    similarities_score, index_score = torch.topk(
        encoding_model.similarity(sentence_embedding, embeddings), k=5
    )
    return [(pairs[i.item()][0], pairs[i.item()][1]) for i in index_score.squeeze(0)]


In [13]:
# sentence = "movies with actions shooting and cars"
# sentence_embedding = model.encode(sentence)
# similarities_score, index_score = torch.topk(
#     model.similarity(sentence_embedding, embeddings), k=5
# )
# [(pairs[i.item()][0], pairs[i.item()][1]) for i in index_score.squeeze(0)]

In [77]:
find_similar_movie(
    "A movie with action and drama, somethign good with a bit of fantasy"
)

[('Lal Dupatta Malmal Ka', 'This is a romantic drama movie.[2]'),
 ('Bless Me, Ultima',
  'A drama set in New Mexico during World War II, centered on the relationship between a young man and an elderly medicine woman who helps him contend with the battle between good and evil that rages in his village.'),
 ('Kalgejje',
  'The film is about romance and has music as its primary element.'),
 ('Age Jodi Jantam Tui Hobi Por',
  'The film is mainly based on the rural background. This movie contains a triangular romance between a young woman and two young men.[3][7]'),
 ('Rasta', 'A fast-paced thriller, with Mithun providing the Twist.')]

In [78]:
class MovieExpertBot:
    def __init__(self, model_id):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, device_map="mps")
        self.instruction = "You are an expert and a highly proclaimed film and tv-series critique and you should know how to recommend movies"
        self.memory = [{"role": "system", "content": self.instruction}]

    def send_message_to_model(self, message):
        message = f"Similar semantic movies with the {find_similar_movie(message)} User question: {message}"
        self.memory.append({"role": "user", "content": message})
        tokenized_chat = self.tokenizer.apply_chat_template(
            self.memory,
            tokenized=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to("mps")
        len_of_prompt = tokenized_chat["input_ids"].shape[1]
        outputs = self.model.generate(
            **tokenized_chat,
            max_length=1024,
            streamer=TextStreamer(
                self.tokenizer, skip_prompt=True, skip_special_tokens=True
            ),
            top_p=0.8,
            do_sample=True,
        )
        answer = self.tokenizer.decode(outputs[0][len_of_prompt:])
        self.memory.append({"role": "assistant", "content": answer})
        return answer


In [79]:
movie_exp_model = MovieExpertBot("Qwen/Qwen3.5-2B")
answer = movie_exp_model.send_message_to_model("Which is the best action movie ? ")
answer

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Among the list provided, **"Param Dharam"** stands out as the clear choice for an **action movie**, despite its title suggesting a comedic or historical context. It is explicitly categorized as an action film, promising high-octane sequences and spectacular stunts.

While **"Santrash"** is a thrilling thriller and **"Walk Like a Dragon"** is a masterpiece of martial arts (often classified under action), and **"Suraj"** is a fan-favorite Mithun movie, the prompt specifically asks for an action movie.

Here is my critique and recommendation:

**The Verdict: Param Dharam**

If you are looking for pure adrenaline, explosive stunts, and the definition of an action spectacle, **"Param Dharam"** is your best bet from this specific selection. It carries the energy of a major Hindi action blockbuster.

**Why it's the winner:**
*   **Genre Alignment:** The prompt explicitly defines it as an "action film."
*   **Stamina:** It is designed for endurance, offering the long sequences of chases and fi

'Among the list provided, **"Param Dharam"** stands out as the clear choice for an **action movie**, despite its title suggesting a comedic or historical context. It is explicitly categorized as an action film, promising high-octane sequences and spectacular stunts.\n\nWhile **"Santrash"** is a thrilling thriller and **"Walk Like a Dragon"** is a masterpiece of martial arts (often classified under action), and **"Suraj"** is a fan-favorite Mithun movie, the prompt specifically asks for an action movie.\n\nHere is my critique and recommendation:\n\n**The Verdict: Param Dharam**\n\nIf you are looking for pure adrenaline, explosive stunts, and the definition of an action spectacle, **"Param Dharam"** is your best bet from this specific selection. It carries the energy of a major Hindi action blockbuster.\n\n**Why it\'s the winner:**\n*   **Genre Alignment:** The prompt explicitly defines it as an "action film."\n*   **Stamina:** It is designed for endurance, offering the long sequences of

In [41]:
answer = movie_exp_model.send_message_to_model("But what about movie with Joker ?")

If we shift the criteria from pure **kinetic spectacle** (which *Mad Max* represents) to **cultural mythology, psychological depth, and the fusion of art-house drama with high-stakes conflict**, the inclusion of **The Joker** changes the landscape entirely.

The Joker is not just an action hero; he is a philosophical antagonist, a tragic figure, and a cultural phenomenon. A movie featuring him cannot simply be an "action film"; it must be a **drama with action** or a **dark comedy with action**. The genre is no longer just "fighting"; it is "exploring the nature of sanity, morality, and chaos."

Here is the critique of the movies that have redefined the genre with the Joker, depending on what kind of "best" you are looking for:

### 1. **The King of Psychological Action:** *The Dark Knight* (2008)
If you define "best" as the perfect blend of action, drama, and the Joker's most iconic transformation, this is the undisputed winner.

*   **The Unique Selling Point:** In almost every actio

In [80]:
movie_exp_model.memory

[{'role': 'system',
  'content': 'You are an expert and a highly proclaimed film and tv-series critique and you should know how to recommend movies'},
 {'role': 'user',
  'content': 'Similar semantic movies with the [(\'Param Dharam\', \'Param Dharam is an action film.\'), (\'Laparwah\', \'Laparwah is an out-and-out action film.\'), (\'Santrash\', \'A fast-paced thriller, with Mithun in the lead.\'), (\'Walk Like a Dragon\', "Film\'s introduction:"), (\'Suraj\', \'Suraj is an Action film for Mithun Fans.\')] User question: Which is the best action movie ? '},
 {'role': 'assistant',
  'content': 'Among the list provided, **"Param Dharam"** stands out as the clear choice for an **action movie**, despite its title suggesting a comedic or historical context. It is explicitly categorized as an action film, promising high-octane sequences and spectacular stunts.\n\nWhile **"Santrash"** is a thrilling thriller and **"Walk Like a Dragon"** is a masterpiece of martial arts (often classified und